# 00 · Sanity checks for the evaluation and submission code
Runs on CPU in ~1-2 min. Verifies before any training that:
1. `src/eval.py` gives mAP ≈ 1.0 when the val ground truth is fed back as predictions (checks the 0→1-based id shift, box format and ignore handling);
2. `src/submission.py` writes CSVs (empty and random boxes) that pass the validator.

In [1]:
# Paths: auto-detected for Kaggle and for a local clone (data in <repo>/data, outputs in <repo>/outputs).
# Override any of the three variables by hand if your layout differs.
from pathlib import Path
if Path("/kaggle/input").exists():
    DATA_ROOT = next(Path("/kaggle/input").glob("**/test_images.csv")).parent  # competition folder
    REPO_DIR  = Path("/kaggle/working/Video-Detection-HW2")                     # git clone of this repo
    WORK_DIR  = Path("/kaggle/working/outputs")
else:
    REPO_DIR  = Path.cwd().parent if Path.cwd().name == "notebooks" else Path.cwd()
    DATA_ROOT = REPO_DIR / "data"
    WORK_DIR  = REPO_DIR / "outputs"
assert (DATA_ROOT / "test_images.csv").exists(), f"test_images.csv not found in {DATA_ROOT}"
print("DATA_ROOT:", DATA_ROOT, "| REPO_DIR:", REPO_DIR, "| WORK_DIR:", WORK_DIR)

DATA_ROOT: /Users/anastasiiayablunovska/year3/ai/Video-Detection-HW2/data | REPO_DIR: /Users/anastasiiayablunovska/year3/ai/Video-Detection-HW2 | WORK_DIR: /Users/anastasiiayablunovska/year3/ai/Video-Detection-HW2/outputs


In [2]:
# Install the pinned dependencies once per session (torch/torchvision are already on Kaggle/Colab).
# !git clone <your-repo-url> {REPO_DIR}
# !pip install -q -r {REPO_DIR}/requirements.txt
# Colab only: keep WORK_DIR on Google Drive so it survives disconnects
# from google.colab import drive; drive.mount("/content/drive")

In [3]:
# Make `src/` importable and print library versions (reproducibility)
import sys
from pathlib import Path
sys.path.insert(0, str(Path(REPO_DIR).resolve()))

from src.utils import load_config, seed_everything, print_versions, detect_env, ensure_dir
print("env:", detect_env())
print_versions()

env: local
python          3.12.6
numpy           2.4.6
torch           2.11.0
torchvision     0.26.0
ultralytics     8.4.208
faster_coco_eval 1.8.0
pycocotools     installed
ensemble_boxes  installed
sahi            0.14.0
cv2             5.0.0
wandb           0.31.0
device          CPU only


In [4]:
# Load the shared config, fix seeds, create output folders
cfg = load_config(Path(REPO_DIR) / "configs" / "base.yaml")
seed_everything(cfg["seed"])

DATA_ROOT, WORK_DIR = Path(DATA_ROOT), ensure_dir(WORK_DIR)
FIG_DIR = ensure_dir(Path(REPO_DIR) / "results" / "figures")
print("train step:", cfg["data"]["train_frame_step"], "| work dir:", WORK_DIR)

train step: 3 | work dir: /Users/anastasiiayablunovska/year3/ai/Video-Detection-HW2/outputs


## 1. Evaluation: ground truth as predictions

In [5]:
# Build canonical predictions [x1,y1,x2,y2,score,cls0] from the val YOLO labels (score = 1)
import numpy as np
from src.dataset import build_frame_index, read_yolo_labels

val_idx = build_frame_index(DATA_ROOT / "val", frame_step=1)
gt_preds = {}
for r in val_idx.itertuples():
    boxes, cls = read_yolo_labels(r.label_path, r.width, r.height)
    gt_preds[r.key] = np.concatenate([boxes, np.ones((len(boxes), 1)), cls[:, None]], axis=1).astype(np.float32)
print(len(gt_preds), "frames,", sum(len(v) for v in gt_preds.values()), "boxes")

[index] val: 7 clips, 2846 frames (step=1)
2846 frames, 114132 boxes


In [6]:
# Evaluate: expect mAP close to 1.0 (small loss only from YOLO-label rounding)
from src.eval import evaluate, per_class_table
VAL_JSON = DATA_ROOT / "val" / "annotations_coco.json"
m = evaluate(gt_preds, VAL_JSON, max_dets=cfg["eval"]["max_dets"])
per_class_table(m)

[eval:faster_coco_eval] mAP=1.0000 AP50=1.0000 AP75=1.0000 APs=1.0000 APm=1.0000 APl=1.0000


,class,AP
0,person,1.0
1,bicycle,1.0
2,car,1.0
3,van,1.0
4,truck,1.0
5,tricycle,1.0
6,bus,1.0
7,motor,1.0


In [7]:
# Ignore-region check: one extra false box per frame, placed in the middle of an ignore region.
# With correct ignore handling the score should stay ~unchanged; without it mAP would drop.
from src.dataset import read_ignore
noisy = {k: v.copy() for k, v in gt_preds.items()}
n_added = 0
for r in val_idx.itertuples():
    ign = read_ignore(r.ignore_path, r.width, r.height)
    if len(ign):
        x1, y1, x2, y2 = ign[0]
        cx, cy, s = (x1 + x2) / 2, (y1 + y2) / 2, max(2.0, min(x2 - x1, y2 - y1) / 4)
        fake = np.array([[cx - s, cy - s, cx + s, cy + s, 1.0, 0]], np.float32)
        noisy[r.key] = np.concatenate([noisy[r.key], fake])
        n_added += 1
print("fake boxes inside ignore regions:", n_added)
m_noisy = evaluate(noisy, VAL_JSON, max_dets=cfg["eval"]["max_dets"])

fake boxes inside ignore regions: 1434
[eval:faster_coco_eval] mAP=1.0000 AP50=1.0000 AP75=1.0000 APs=1.0000 APm=1.0000 APl=1.0000


In [8]:
# Control: the same kind of fake box OUTSIDE ignore regions (5x5 px top-left corner) must LOWER mAP,
# otherwise the check above proves nothing
outside = {k: np.concatenate([v, np.array([[0, 0, 5, 5, 1.0, 0]], np.float32)]) for k, v in gt_preds.items()}
m_out = evaluate(outside, VAL_JSON, max_dets=cfg["eval"]["max_dets"])
assert m_noisy["mAP"] > m_out["mAP"], "ignore regions are not handled - fake boxes inside them are penalised"
print(f"inside ignore: {m_noisy['mAP']:.4f}  vs  outside: {m_out['mAP']:.4f}  -> ignore handling OK")

[eval:faster_coco_eval] mAP=0.9967 AP50=0.9967 AP75=0.9967 APs=0.9843 APm=1.0000 APl=1.0000
inside ignore: 1.0000  vs  outside: 0.9967  -> ignore handling OK


## 2. Submission writer + validator

In [9]:
# An empty submission (no detections) written by our code must pass the validator
from src.submission import make_submission, validate_submission
TEST_CSV = DATA_ROOT / "test_images.csv"
empty_csv = WORK_DIR / "submissions" / "empty.csv"
make_submission({}, TEST_CSV, empty_csv)
validate_submission(empty_csv, TEST_CSV)

[submission] wrote /Users/anastasiiayablunovska/year3/ai/Video-Detection-HW2/outputs/submissions/empty.csv | rows=3811 | images without preds=3811
[validate] OK | rows=3811 | boxes=0 | mean boxes/img=0.0


True

In [10]:
# Round trip with random boxes (some partly outside the image -> must be clipped by the writer)
import pandas as pd
rng = np.random.default_rng(cfg["seed"])
test_df = pd.read_csv(TEST_CSV)
fake = {}
for r in test_df.itertuples():
    n = rng.integers(0, 700)                      # > 500 on purpose: writer must keep top-500
    xy = rng.uniform(-20, [r.width, r.height], size=(n, 2))
    wh = rng.uniform(5, 80, size=(n, 2))
    fake[r.image_id] = np.concatenate([xy, xy + wh, rng.uniform(0, 1, (n, 1)),
                                       rng.integers(0, 8, (n, 1))], axis=1).astype(np.float32)
out_csv = WORK_DIR / "submissions" / "smoke_test.csv"
make_submission(fake, TEST_CSV, out_csv, max_det=cfg["submission"]["max_det"],
                conf_thres=cfg["submission"]["conf_thres"], decimals=cfg["submission"]["decimals"])
validate_submission(out_csv, TEST_CSV)

[submission] wrote /Users/anastasiiayablunovska/year3/ai/Video-Detection-HW2/outputs/submissions/smoke_test.csv | rows=3811 | images without preds=0
[validate] OK | rows=3811 | boxes=1200767 | mean boxes/img=315.1


True

In [11]:
# Check that test file names resolve to real images (paths in test_images.csv)
from src.dataset import resolve_test_path
print(test_df.head(3))
print(resolve_test_path(DATA_ROOT, test_df.file_name.iloc[0]))

           image_id                         file_name      clip  frame  width  \
0  clip_001_0000001  test/images/clip_001/0000001.jpg  clip_001      1   1920   
1  clip_001_0000002  test/images/clip_001/0000002.jpg  clip_001      2   1920   
2  clip_001_0000003  test/images/clip_001/0000003.jpg  clip_001      3   1920   

   height  
0    1080  
1    1080  
2    1080  
/Users/anastasiiayablunovska/year3/ai/Video-Detection-HW2/data/test/images/clip_001/0000001.jpg
